In [1]:
!pip install ipython-sql

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 1.3 MB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 1.7 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 1.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.8 MB/s  0:00:01

   ---------------------------------------- 0/6 [ipython-genutils]
   ------ --------------------------------- 1/6 [sqlparse]
   ------ --------------------------------- 1/6 [sqlparse]
   ------ --------------------------------- 1/6 [sqlparse]
   ------ --------------------------------- 1/6 [sqlparse]
   ------------- -------------------------- 2/6 [prettytable]
   -------------------- ------------------- 3/6 [greenlet]
   -------------------- ------------------- 3/6 [greenlet]
 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd

In [5]:
import sqlite3

In [10]:
# Load the CSV into a pandas dataframe
df = pd.read_csv('data/insurance.csv')

In [8]:
import os
os.getcwd()

'C:\\Users\\YASHASWINI SANDEEP\\OneDrive\\Desktop\\Projects\\BI_Portfolio\\SQL_Analysis'

In [9]:
os.listdir()

['.ipynb_checkpoints',
 'data',
 'insurance_analysis.ipynb',
 'outputs',
 'queries',
 'README.md']

In [11]:
# Preview the data
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [12]:
# Create a SQLite database in momory
conn = sqlite3.connect('insurance.db')

In [17]:
#Load the dataframe into SQLite as a table called 'insurance'
df.to_sql('insurance', conn, if_exists='replace', index=False)

print("Database ready!")

Database ready!


In [18]:
# Let's verify the table exists
pd.read_sql_query("SELECT * FROM insurance LIMIT 5", conn)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [19]:
# Query 1: Total and Average Charges by Region
query1 = """
SELECT
    region,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charge,
    ROUND(SUM(charges), 2) AS total_charges
FROM insurance
GROUP BY region
ORDER BY total_charges DESC;
"""

pd.read_sql_query(query1, conn)

,region,total_customers,avg_charge,total_charges
0,southeast,364,14735.41,5363689.76
1,northeast,324,13406.38,4343668.58
2,northwest,325,12417.58,4035712.00
3,southwest,325,12346.94,4012754.65


In [20]:
# Query 2: Average BMI and charges - Smokers vs Non-Smokers
query2 = """
SELECT
    smoker,
    COUNT(*) AS total_customers,
    ROUND(AVG(bmi), 2) AS avg_bmi,
    ROUND(AVG(charges), 2) AS avg_charges,
    ROUND(MIN(charges), 2) AS min_charges,
    ROUND(MAX(charges), 2) AS max_charges
FROM insurance
GROUP BY smoker
ORDER BY avg_charges DESC;
"""

pd.read_sql_query(query2, conn)
    

,smoker,total_customers,avg_bmi,avg_charges,min_charges,max_charges
0,yes,274,30.71,32050.23,12829.46,63770.43
1,no,1064,30.65,8434.27,1121.87,36910.61


In [23]:
# Query 3: Top 10 Highest Claims
query3 = """
SELECT
    age,
    sex,
    bmi,
    smoker,
    region,
    ROUND(charges, 2) AS charges
FROM insurance
ORDER BY charges DESC
LIMIT 10;
"""

pd.read_sql_query(query3, conn)

,age,sex,bmi,smoker,region,charges
0,54,female,47.410,yes,southeast,63770.43
1,45,male,30.360,yes,southeast,62592.87
2,52,male,34.485,yes,northwest,60021.40
3,31,female,38.095,yes,northeast,58571.07
4,33,female,35.530,yes,northwest,55135.40
5,60,male,32.800,yes,southwest,52590.83
6,28,male,36.400,yes,southwest,51194.56
7,64,male,36.960,yes,southeast,49577.66
8,59,male,41.140,yes,southeast,48970.25
9,44,female,38.060,yes,southeast,48885.14


In [24]:
# Query 4: Charges by Age Group
query4 = """
SELECT
    CASE
        WHEN age < 25 THEN 'Young (Under 25)'
        WHEN age BETWEEN 25 AND 40 THEN 'Adult (25-40)'
        WHEN age BETWEEN 41 AND 55 THEN ' Middle Aged (41-55)'
        ELSE 'Senior (55+)'
    END AS age_group,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charges,
    ROUND(SUM(charges), 2) AS total_charges
FROM insurance
GROUP BY age_group
ORDER BY avg_charges DESC;
"""

pd.read_sql_query(query4, conn)

,age_group,total_customers,avg_charges,total_charges
0,Senior (55+),216,18795.99,4059934.66
1,Middle Aged (41-55),421,15515.62,6532074.89
2,Adult (25-40),423,11013.39,4658662.83
3,Young (Under 25),278,9011.34,2505152.61


In [27]:
# Query 5: Rank customers by charges within each region
query5 = """
SELECT * FROM (
    SELECT
        age,
        sex,
        region,
        ROUND(charges, 2) AS charges,
        RANK() OVER (
            PARTITION BY region 
            ORDER BY charges DESC
        ) AS rank_in_region
    FROM insurance
)
WHERE rank_in_region <= 3
ORDER BY region, rank_in_region;
"""

pd.read_sql_query(query5, conn)

,age,sex,region,charges,rank_in_region
0,31,female,northeast,58571.07,1
1,54,male,northeast,48549.18,2
2,61,female,northeast,48517.56,3
3,52,male,northwest,60021.40,1
4,33,female,northwest,55135.40,2
5,58,male,northwest,47496.49,3
6,54,female,southeast,63770.43,1
7,45,male,southeast,62592.87,2
8,64,male,southeast,49577.66,3
9,60,male,southwest,52590.83,1


In [29]:
# Query 6: Impact of number of children on charges
query6 = """
SELECT 
    children,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charges,
    ROUND(AVG(bmi), 2) AS avg_bmi
FROM insurance
GROUP BY children
ORDER BY children DESC;
"""

pd.read_sql_query(query6, conn)

,children,total_customers,avg_charges,avg_bmi
0,5,18,8786.04,29.61
1,4,25,13850.66,31.39
2,3,157,15355.32,30.68
3,2,240,15073.56,30.98
4,1,324,12731.17,30.62
5,0,574,12365.98,30.55


In [32]:
# Query 7: Charges by Gender
query7 = """
SELECT
    sex,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charges,
    ROUND(AVG(bmi), 2) AS avg_bmi,
    ROUND(AVG(age), 2) AS avg_age
FROM insurance
GROUP BY sex
ORDER BY avg_charges DESC;
"""

pd.read_sql_query(query7, conn)

,sex,total_customers,avg_charges,avg_bmi,avg_age
0,male,676,13956.75,30.94,38.92
1,female,662,12569.58,30.38,39.50


In [34]:
# Query 8: Full risk profile - combining all factors
query8 = """
SELECT 
    region,
    smoker,
    CASE
        WHEN age < 25 THEN 'Young'
        WHEN age BETWEEN 25 AND 40 THEN 'Adult'
        WHEN age BETWEEN 41 AND 55 THEN 'Middle Aged'
        ELSE 'Senior'
    END AS age_group,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charges
FROM insurance
GROUP BY region, smoker, age_group
ORDER BY avg_charges DESC
LIMIT 10;
"""

pd.read_sql_query(query8, conn)

,region,smoker,age_group,total_customers,avg_charges
0,southwest,yes,Senior,7,48502.42
1,southeast,yes,Senior,14,40393.78
2,southeast,yes,Middle Aged,33,38285.30
3,northwest,yes,Senior,11,35993.17
4,northeast,yes,Senior,8,34945.64
5,northeast,yes,Middle Aged,21,33572.80
6,southeast,yes,Young,17,32995.33
7,southwest,yes,Adult,25,31891.17
8,northwest,yes,Middle Aged,19,31823.62
9,southwest,yes,Middle Aged,12,30648.65


In [35]:
# Save all query results to Excel for GitHub
with pd.ExcelWriter('outputs/sql_analysis_results.xlsx') as writer:
    pd.read_sql_query(query1, conn).to_excel(writer, sheet_name='Charges by Region', index=False)
    pd.read_sql_query(query2, conn).to_excel(writer, sheet_name='Smokers vs Non-Smokers', index=False)
    pd.read_sql_query(query3, conn).to_excel(writer, sheet_name='Top 10 Claims', index=False)
    pd.read_sql_query(query4, conn).to_excel(writer, sheet_name='Age Group Analysis', index=False)
    pd.read_sql_query(query5, conn).to_excel(writer, sheet_name='Regional Rankings', index=False)
    pd.read_sql_query(query6, conn).to_excel(writer, sheet_name='Children Impact', index=False)
    pd.read_sql_query(query7, conn).to_excel(writer, sheet_name='Gender Analysis', index=False)
    pd.read_sql_query(query8, conn).to_excel(writer, sheet_name='Full Risk Profile', index=False)

print("Results saved to Excel!")


Results saved to Excel!


In [36]:
sql_queries = """
-- Query 1: Total and Average Charges by Region
SELECT 
    region,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charge,
    ROUND(SUM(charges), 2) AS total_charges
FROM insurance
GROUP BY region
ORDER BY total_charges DESC;

-- Query 2: Smokers vs Non-Smokers
SELECT
    smoker,
    COUNT(*) AS total_customers,
    ROUND(AVG(bmi), 2) AS avg_bmi,
    ROUND(AVG(charges), 2) AS avg_charges,
    ROUND(MIN(charges), 2) AS min_charges,
    ROUND(MAX(charges), 2) AS max_charges
FROM insurance
GROUP BY smoker
ORDER BY avg_charges DESC;

-- Query 3: Top 10 Highest Claims
SELECT
    age, sex, bmi, smoker, region,
    ROUND(charges, 2) AS charges
FROM insurance
ORDER BY charges DESC
LIMIT 10;

-- Query 4: Charges by Age Group
SELECT
    CASE 
        WHEN age < 25 THEN 'Young (Under 25)'
        WHEN age BETWEEN 25 AND 40 THEN 'Adult (25-40)'
        WHEN age BETWEEN 41 AND 55 THEN 'Middle Aged (41-55)'
        ELSE 'Senior (55+)'
    END AS age_group,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charges,
    ROUND(SUM(charges), 2) AS total_charges
FROM insurance
GROUP BY age_group
ORDER BY avg_charges DESC;

-- Query 5: Top 3 Customers per Region (Window Function)
SELECT * FROM (
    SELECT
        age, sex, region,
        ROUND(charges, 2) AS charges,
        RANK() OVER (
            PARTITION BY region 
            ORDER BY charges DESC
        ) AS rank_in_region
    FROM insurance
)
WHERE rank_in_region <= 3
ORDER BY region, rank_in_region;

-- Query 6: Impact of Number of Children
SELECT
    children,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charges,
    ROUND(AVG(bmi), 2) AS avg_bmi
FROM insurance
GROUP BY children
ORDER BY children ASC;

-- Query 7: Gender Analysis
SELECT
    sex,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charges,
    ROUND(AVG(bmi), 2) AS avg_bmi,
    ROUND(AVG(age), 2) AS avg_age
FROM insurance
GROUP BY sex
ORDER BY avg_charges DESC;

-- Query 8: Combined Risk Profile
SELECT
    region,
    smoker,
    CASE 
        WHEN age < 25 THEN 'Young'
        WHEN age BETWEEN 25 AND 40 THEN 'Adult'
        WHEN age BETWEEN 41 AND 55 THEN 'Middle Aged'
        ELSE 'Senior'
    END AS age_group,
    COUNT(*) AS total_customers,
    ROUND(AVG(charges), 2) AS avg_charges
FROM insurance
GROUP BY region, smoker, age_group
ORDER BY avg_charges DESC
LIMIT 10;
"""

with open('queries/analysis.sql', 'w') as f:
    f.write(sql_queries)

print("SQL queries saved!")

SQL queries saved!


In [38]:
readme_content = """# Insurance Data - SQL Analysis

## Project Overview
Health insurance companies face a core business challenge: accurately predicting 
how much a customer is likely to cost them in medical expenses. Charge too little 
in premiums and the company loses money. Charge too much and customers leave.

This project uses SQL to analyse a real-world health insurance dataset of 1,338 
customers to uncover which factors — age, BMI, smoking status, region, and family 
size — have the strongest influence on annual medical charges.

The analysis simulates the kind of exploratory data work a Business Intelligence 
team at an insurance company would perform to support pricing decisions, risk 
segmentation, and regional strategy.

## Business Questions Answered
1. Which region generates the highest insurance charges?
2. How much does smoking impact the cost of insurance?
3. Do older customers cost significantly more than younger ones?
4. Who are the highest-risk customers in each region?
5. Does having children or being a certain gender affect charges?

## Project Structure

SQL_Analysis/
├── data/                          # Raw dataset (not uploaded to GitHub)
├── outputs/
│   └── sql_analysis_results.xlsx  # All query results exported to Excel
├── queries/
│   └── analysis.sql               # All SQL queries in plain .sql file
├── insurance_analysis.ipynb       # Main Jupyter Notebook
└── README.md                      # Project documentation


## Dataset
- Source: Kaggle - Medical Cost Personal Dataset
- Link: https://www.kaggle.com/datasets/mirichoi0218/insurance
- Size: 1,338 records
- Columns: age, sex, bmi, children, smoker, region, charges

## Analysis Performed

| Query | Analysis | SQL Concepts Used |
|-------|----------|-------------------|
| 1 | Total and Average Charges by Region | GROUP BY, SUM, AVG, COUNT, ROUND |
| 2 | Smokers vs Non-Smokers Comparison | GROUP BY, MIN, MAX |
| 3 | Top 10 Highest Claims | ORDER BY, LIMIT |
| 4 | Charges by Age Group | CASE WHEN |
| 5 | Top 3 Customers per Region | Window Functions, RANK(), Subquery |
| 6 | Impact of Number of Children | GROUP BY on numerical variable |
| 7 | Gender Based Analysis | Multi-metric comparison |
| 8 | Combined Risk Profile | Multi-dimensional GROUP BY |

## Key Insights
1. Smoking is the single biggest cost driver - smokers pay nearly 4x more ($32,050 vs $8,434)
2. Southeast is the most expensive region - highest total and average charges
3. Age increases charges steadily - seniors pay 2x more than young customers
4. Every single top 10 highest claim belongs to a smoker, without exception
5. Males pay slightly more than females ($13,956 vs $12,569)

## Tools Used
- Python 3
- Pandas
- SQLite3
- Jupyter Notebook

## How to Run
1. Clone this repository
2. Download the dataset from Kaggle and place it in the data/ folder
3. Open insurance_analysis.ipynb in Jupyter Notebook
4. Run all cells from top to bottom

## Author
Yashaswini Sandeep
"""

# Save the README
with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("README saved!")

README saved!
